In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import mudata as md
import cell2net as cn
from tqdm import tqdm
import pandas as pd
md.set_options(pull_on_update=False)

In [ ]:
data_dir = "./02_prepare_data/mdata.h5mu"
in_dir = "./03_train_cell2net"
out_dir = "./05_interpretation_tf"

os.makedirs(out_dir, exist_ok=True)

In [3]:
mdata_bulk = md.read_h5mu(data_dir)

In [4]:
genes = mdata_bulk.uns['peak_to_gene']['gene'].unique().tolist()

In [5]:
len(genes)

1927

In [6]:
for gene in tqdm(genes):
    if os.path.exists(f"{out_dir}/{gene}.npy"):
        continue
    
    model = cn.pd.model.Cell2Net(mdata=mdata_bulk, 
                                 gene=gene, 
                                 covariates=['total_counts_rna_log', 'total_counts_atac_log'])

    model.load(dir_path=f"{in_dir}/model")
    model.to_device('cuda:0')
    
    tf_attr = cn.ip.tf_attr(model, 
                            batch_size=2,
                            n_steps=100,
                            multiply_by_inputs=True)
    
    df = cn.ip.tf_to_gene(model.mdata, tf_attr, groupby="cell_type_v2", n_tfs=10)
    np.save(f"{out_dir}/{gene}.npy", tf_attr)
    df.to_csv(f"{out_dir}/{gene}.csv", index=False)

100%|██████████| 1927/1927 [00:00<00:00, 4783.93it/s]


In [12]:
df_list = []
for gene in genes:
    df = pd.read_csv(f"{out_dir}/{gene}.csv")

    df_list.append(df)
df_p2g = pd.concat(df_list).reset_index(drop=True)

In [13]:
df = pd.concat(df_list)

In [14]:
df

,tf,gene,cell_type_v2,mean_attr,std_attr
0,KLF2,ISG15,B cell,1.415844,0.468231
1,ELF1,ISG15,B cell,0.975179,0.298402
2,MEF2C,ISG15,B cell,0.851608,0.448217
3,IKZF3,ISG15,B cell,0.699268,0.239014
4,JUNB,ISG15,B cell,0.691353,0.323894
...,...,...,...,...,...
75,ARID3A,CLIC2,pDC,0.167850,0.062825
76,MAX,CLIC2,pDC,0.154308,0.050194
77,CREB3L2,CLIC2,pDC,0.151772,0.049553
78,MGA,CLIC2,pDC,0.150544,0.062157


In [15]:
len(df['gene'].unique())

1927

In [16]:
df.to_csv(f"{out_dir}/tf_to_gene.csv")